In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install dspy-ai

In [ ]:
import threading
import subprocess

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
import dspy
import requests

In [ ]:
def setup_ollama():
  try:
    requests.get("http://localhost:11434/api/tags")
    print("Ollama is running")
  except:
    print("Ollama not running. Start with: ollama serve")
    return False
setup_ollama()

In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
dspy.settings.configure(
    lm=dspy.LM(
      model="ollama/qwen2.5:3b",
      api_base="http://localhost:11434",
      max_tokens=500,
      temperature=0.7,
    )
)

In [ ]:
import sqlite3

def create_biblioteca_database():
  conn = sqlite3.connect("biblioteca.db")
  c = conn.cursor()

  c.execute("""CREATE TABLE IF NOT EXISTS autores (
      id INTEGER PRIMARY KEY, nome TEXT, nacionalidade TEXT)""")
      
  c.execute("""CREATE TABLE IF NOT EXISTS livros (
      id INTEGER PRIMARY KEY, titulo TEXT, autor_id INTEGER, ano_publicacao INTEGER, genero TEXT)""")
      
  c.execute("""CREATE TABLE IF NOT EXISTS emprestimos (
      id INTEGER PRIMARY KEY, livro_id INTEGER, nome_usuario TEXT, data_emprestimo TEXT, devolvido BOOLEAN)""")

  c.executemany("INSERT OR IGNORE INTO autores VALUES (?, ?, ?)", [
    (1, "Machado de Assis", "Brasileira"),
    (2, "J.R.R. Tolkien", "Britânica"),
    (3, "Agatha Christie", "Britânica")
  ])

  c.executemany("INSERT OR IGNORE INTO livros VALUES (?, ?, ?, ?, ?)", [
    (1, "Dom Casmurro", 1, 1899, "Romance"),
    (2, "O Senhor dos Anéis", 2, 1954, "Fantasia"),
    (3, "O Hobbit", 2, 1937, "Fantasia"),
    (4, "Assassinato no Expresso do Oriente", 3, 1934, "Mistério")
  ])

  c.executemany("INSERT OR IGNORE INTO emprestimos VALUES (?, ?, ?, ?, ?)", [
    (1, 2, "Lucas", "2023-10-01", False),
    (2, 4, "Ana", "2023-10-05", True),
    (3, 1, "Marcos", "2023-10-10", False)
  ])

  conn.commit()
  conn.close()
  print("Database Biblioteca criado")

create_biblioteca_database()

In [ ]:
class GenerateSQL(dspy.Signature):
  """Generate SQL from natural language. Respond ONLY with the reasoning and the valid SQL query. Do not add any conversational text or formatting.

  Database schema:
  - autores: id, nome, nacionalidade
  - livros: id, titulo, autor_id, ano_publicacao, genero
  - emprestimos: id, livro_id, nome_usuario, data_emprestimo, devolvido
  """

  question = dspy.InputField(desc="Natural language question")
  sql_query = dspy.OutputField(desc="Valid SQL query")

In [ ]:
class SQLGenerator(dspy.Module):
  def __init__(self):
    super().__init__()
    self.generator = dspy.ChainOfThought(GenerateSQL)
    self.refiner = dspy.ChainOfThought(
      "question, sql_query, error -> refined_sql"
    )

  def forward(self, question, conn):
    output = self.generator(question=question)
    sql = output.sql_query.strip()
    
    # Remove markdown caso exista
    sql = sql.replace("```sql", "").replace("```", "").strip()

    try:
      results = conn.execute(sql).fetchall()
      return dspy.Prediction(sql_query=sql, results=results, error=None)
    except Exception as e:
      refined = self.refiner(question=question, sql_query=sql, error=str(e))
      refined_sql = refined.refined_sql.replace("```sql", "").replace("```", "").strip()
      try:
        results = conn.execute(refined_sql).fetchall()
        return dspy.Prediction(sql_query=refined_sql, results=results, error=None)
      except Exception as e2:
        return dspy.Prediction(sql_query=refined_sql, results=None, error=str(e2))

In [ ]:
def create_examples():
  return [
    dspy.Example(
      question="Quantos livros o autor J.R.R. Tolkien escreveu na base?",
      reasoning="Preciso fazer JOIN entre livros e autores e contar onde o nome do autor é J.R.R. Tolkien.",
      sql_query="SELECT COUNT(*) FROM livros l JOIN autores a ON l.autor_id = a.id WHERE a.nome = 'J.R.R. Tolkien'"
    ).with_inputs("question"),
    dspy.Example(
      question="Qual o livro mais antigo cadastrado?",
      reasoning="Devo ordenar os livros pelo ano de publicacao em ordem crescente e pegar apenas o primeiro registro.",
      sql_query="SELECT titulo, ano_publicacao FROM livros ORDER BY ano_publicacao ASC LIMIT 1"
    ).with_inputs("question"),
    dspy.Example(
      question="Liste os usuários que ainda não devolveram os livros",
      reasoning="Selecionar os nomes de usuarios da tabela emprestimos filtrando os que não foram devolvidos.",
      sql_query="SELECT nome_usuario FROM emprestimos WHERE devolvido = False"
    ).with_inputs("question")
  ]

generator = SQLGenerator()
generator.generator.demos = create_examples()
print("✓ Generator configured with few-shot examples")

In [ ]:
conn = sqlite3.connect("biblioteca.db")
test_questions = [
    "Quantos autores são do Brasil?",
    "Qual é o ano de publicação de O Hobbit?",
    "Quem emprestou o livro Assassinato no Expresso do Oriente?",
    "Quais livros estão atualmente emprestados?",
]

for i, question in enumerate(test_questions, 1):
    print(f"Query {i}: {question}")
    result = generator(question=question, conn=conn)
    print(f"SQL: {result.sql_query}")
    print(f"Results: {result.results if not result.error else f' {result.error}'}\n")